# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the mlcroissant dataset
dataset = mlc.Dataset(url)

# Access dataset metadata
meta = dataset.metadata

print(f"Dataset: {meta.name}\nDescription: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record set @ids, names, and their field definitions
record_sets = meta.record_sets

print(f"Number of Record Sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet: @id = {rs.id}")
    print(f"  Name: {rs.name}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields (by @id):")
        for field in rs.fields:
            print(f"    - {field.id} (name: {getattr(field, 'name', 'N/A')})")
    print("-")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We'll extract all dataframes, keyed by record set @id
# List record_set @ids:
record_set_ids = [rs.id for rs in meta.record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from RecordSet: {record_set_id}")
        print(f"Columns: {df.columns.tolist()[:5]}{'...' if len(df.columns)>5 else ''}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# For demonstration, pick the first RecordSet if available
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"\nColumns in DataFrame for {example_record_set_id}:\n{dataframes[example_record_set_id].columns.tolist()}")
    display(dataframes[example_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# We'll use the first record set and choose a numeric field
record_set_id = example_record_set_id
df = dataframes[record_set_id]

# Try to pick a common numeric field, fallback if not present
numeric_candidates = ['cr:coverage', 'cr:peptide_count', 'cr:MW', 'cr:abundance', 'cr:pI', 'cr:unique_peptides']
numeric_field = None

for col in numeric_candidates:
    if col in df.columns:
        numeric_field = col
        break

if numeric_field is None:
    print("No suitable numeric field found for EDA. Available columns:", df.columns.tolist())
else:
    # Convert numeric field to float (if not already)
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)}/ {len(df)}")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a field, such as description or accession
    group_candidates = ['cr:description', 'cr:accession', 'cr:sample']
    group_field = None
    for col in group_candidates:
        if col in df.columns:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
        print(f"\nTop grouped by {group_field}:")
        display(grouped_df.head())
    else:
        print("No suitable grouping field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If a group field exists, plot top 10 groups
    if group_field and group_field in df.columns:
        grouped = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)[:10]
        plt.figure(figsize=(10,4))
        sns.barplot(x=grouped.values, y=grouped.index)
        plt.title(f"Top 10 mean {numeric_field} by {group_field}")
        plt.xlabel(f"Mean {numeric_field}")
        plt.ylabel(group_field)
        plt.show()
else:
    print("No suitable numeric field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset was successfully loaded from the Croissant schema and record sets explored.
- Numeric fields such as coverage, peptide count, or molecular weight can be filtered and normalized for downstream analysis.
- Grouping and basic visualizations reveal patterns across proteins, accession records, or samples (when available).
- For deeper use, consult the Croissant schema available at the provided URL for authoritative field `@id`s and further data relationships.